In [27]:
from data_loader import SegmentationDataset, SegmentationDataModule
from evaluation_functions import compute_miou
from perturbation_methods import *
import torch

In [ ]:
# Load dataset

# ESA-PhiLab-Edge/OEOBench-Burnt_Area_Dataset dataset
root_dir = '/local/s3167445/data'
dm = SegmentationDataModule(root_dir, batch_size=2, num_workers=4, transform=None)
dm.setup(stage='test')  # load test dataset

# Detect input channels dynamically
sample_img, _ = dm.test_dataset[0]
in_ch = sample_img.shape[0]  # CHW format
print("Detected input channels:", in_ch)

Detected input channels: 7


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pynas = torch.jit.load('models/PNAS_NVIDIA_jetson_AGX_orin.pt', map_location=device)
pynas.to(device) 
pynas.eval()

RecursiveScriptModule(
  original_name=GenericUNetNetwork
  (encoder): RecursiveScriptModule(
    original_name=ModuleList
    (0): RecursiveScriptModule(original_name=Dropout)
    (1): RecursiveScriptModule(
      original_name=AvgPool
      (0): RecursiveScriptModule(original_name=AvgPool2d)
    )
    (2): RecursiveScriptModule(
      original_name=ConvSE
      (0): RecursiveScriptModule(
        original_name=ConvBnAct
        (0): RecursiveScriptModule(original_name=Conv2d)
        (1): RecursiveScriptModule(original_name=BatchNorm2d)
        (2): RecursiveScriptModule(original_name=ReLU)
      )
      (1): RecursiveScriptModule(
        original_name=SEBlock
        (avg_pool): RecursiveScriptModule(original_name=AdaptiveAvgPool2d)
        (fc): RecursiveScriptModule(
          original_name=Sequential
          (0): RecursiveScriptModule(original_name=Linear)
          (1): RecursiveScriptModule(original_name=ReLU)
          (2): RecursiveScriptModule(original_name=Linear)
      

In [ ]:
from torch.utils.data import DataLoader

temp_ds = dm.test_dataset
temp_loader = DataLoader(temp_ds, batch_size=2, num_workers=4)

current_miou = compute_miou(pynas, temp_loader,num_classes=4,device=device)
print(current_miou)




0.858401358127594


In [ ]:
from models.my_baseline_models import ResNet18UNet

ModuleNotFoundError: No module named 'benchmark'

In [ ]:
from torch.utils.data import DataLoader
from evaluation_functions import compute_miou

resnet = ResNet18UNet(in_channels=7, num_classes=4) 
resnet.to(device)  
resnet.eval()
state_dict = torch.load("models/resnet18.pt", map_location=device)
resnet.load_state_dict(state_dict)
temp_ds = dm.test_dataset
temp_loader = DataLoader(temp_ds, batch_size=2, num_workers=4)

current_miou = compute_miou(resnet, temp_loader,num_classes=4,device=device)
print(current_miou)


NameError: name 'ResNet18UNet' is not defined